# Notebook 4.1  From scores to transcripts: Arabic ASR foundations

**Companion to Chapter 4, *Introduction to Arabic Speech Technologies*.**

**Goal.** Make the chapter runnable: score and normalize Arabic transcripts (Word/Character
Error Rate), see the tokenization trade-off, run the toy forward/Viterbi example, and inspect
Kneser-Ney continuation counts. Pure Python; no downloads.

## 1. Word/Character Error Rate with Arabic normalization

Reproduces the Section 4.2 worked example: literal scoring gives 50% WER; after normalization
the strings match and WER is 0%.

In [ ]:
def _levenshtein(ref, hyp):
    """Token-level edit distance with operation counts (S, D, I)."""
    n, m = len(ref), len(hyp)
    d = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1): d[i][0] = i
    for j in range(m+1): d[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1]+cost)
    return d[n][m]

def error_rate(ref_tokens, hyp_tokens):
    if len(ref_tokens) == 0:
        return 0.0 if len(hyp_tokens) == 0 else 1.0
    return _levenshtein(ref_tokens, hyp_tokens) / len(ref_tokens)

def wer(ref, hyp):
    return error_rate(ref.split(), hyp.split())

def cer(ref, hyp):
    return error_rate(list(ref.replace(" ", "")), list(hyp.replace(" ", "")))

In [ ]:
import re
# Arabic text normalization for scoring. Each rule is OPTIONAL and is a SCORING
# convention, not a claim of linguistic equivalence. Toggle the flags to see the
# effect on Word/Character Error Rate.
TASHKEEL = re.compile(r"[ؗ-ًؚ-ْٰـ]")  # diacritics + Tatweel

def normalize_ar(text, strip_diacritics=True, unify_alef=True,
                 ta_marbuta_to_ha=True, alef_maqsura_to_ya=True, unify_digits=True):
    if strip_diacritics:
        text = TASHKEEL.sub("", text)
    if unify_alef:
        text = re.sub("[آأإٱ]", "ا", text)  # آأإ ٱ -> ا
    if ta_marbuta_to_ha:
        text = text.replace("ة", "ه")  # ة -> ه
    if alef_maqsura_to_ya:
        text = text.replace("ى", "ي")  # ى -> ي
    if unify_digits:
        ar_digits = "٠١٢٣٤٥٦٧٨٩"
        text = text.translate({ord(a): str(i) for i, a in enumerate(ar_digits)})
    return re.sub(r"\s+", " ", text).strip()

In [ ]:
ref = 'ذهب الطالب إلى المدرسة'      # reference
hyp = 'ذهب الطالب الى المدرسه'      # hypothesis: Alef and Ta-marbuta variants
print('raw  WER: {:.0f}%'.format(100*wer(ref, hyp)))
print('norm WER: {:.0f}%'.format(100*wer(normalize_ar(ref), normalize_ar(hyp))))
print('norm CER: {:.0f}%'.format(100*cer(normalize_ar(ref), normalize_ar(hyp))))

## 2. Tokenization trade-off

One sentence, several tokenizations. Counts illustrate why subwords are the usual default for
Arabic: words risk out-of-vocabulary (OOV), characters give long sequences.

In [ ]:
sentence_words = ['سيكتبونها', 'للمكتبة']   # sa-yaktubuunahaa lil-maktaba
chars = [c for w in sentence_words for c in w]
bpe   = ['sy','ktbwn','ha','ll','mktba']                 # illustrative subwords
morph = ['sa-','yaktubuun','-ha','li-','al-','maktaba']   # morphemes
for name, toks in [('words', sentence_words), ('characters', chars),
                   ('BPE subwords', bpe), ('morphemes', morph)]:
    print(f'{name:14} count={len(toks):2}  {toks}')

## 3. The toy forward algorithm and Viterbi

This matches the optional **Going Deeper** box in the chapter: a 2-state HMM over 3 frames. Keep a running score per state. For frame 1 it is the start probability times the emission. For each later frame, update every state in four steps: **(a)** take the previous frame's scores, **(b)** multiply by the probability of moving into the state, **(c)** add the incoming paths together, **(d)** multiply by the emission for this frame. Summing the final scores gives `P(observations)`. **Viterbi** is the same sweep but keeps the single best incoming path (`max` instead of `sum`), which recovers the most likely alignment. You do not need this to follow the chapter; it is here for the curious.

In [ ]:
import numpy as np
states = ['s1', 's2']
pi = np.array([0.6, 0.4])                         # start probability of each state
A  = np.array([[0.7, 0.3], [0.4, 0.6]])           # A[i, j] = P(move to state j | in state i)
B  = np.array([[0.5, 0.1], [0.4, 0.2], [0.1, 0.5]])  # B[t, j] = P(frame t | state j); 3 frames

# ---- Forward algorithm: the four steps from the chapter box ----
alpha = pi * B[0]                 # frame 1: start probability x emission
trellis = [alpha.copy()]
for t in range(1, len(B)):
    incoming = alpha @ A          # (a) previous scores, (b) x transition, (c) summed over paths
    alpha = incoming * B[t]       # (d) x emission for this frame
    trellis.append(alpha.copy())

print('forward trellis (one row per frame, columns = s1, s2):')
for t, row in enumerate(trellis):
    print(f'  frame {t+1}:', np.round(row, 5))
print('P(observations) = sum of the last row =', round(alpha.sum(), 5))

# ---- Viterbi: same sweep, but keep the single best incoming path (max, not sum) ----
delta = pi * B[0]; back = []
for t in range(1, len(B)):
    paths = delta[:, None] * A    # score of every incoming path into each state
    back.append(paths.argmax(0))  # remember the best predecessor
    delta = paths.max(0) * B[t]   # keep the best (max) instead of summing
best = [int(delta.argmax())]
for b in reversed(back):
    best.append(int(b[best[-1]]))
best = [states[i] for i in reversed(best)]
print('viterbi best path =', best, ' prob =', round(delta.max(), 5))


## 4. Kneser-Ney intuition: continuation counts

A word that follows **many different** words is a better fallback in a new context than a word
with high raw frequency that follows only one word.

In [ ]:
corpus = [
    'عبد الرحمن', 'عبد الله', 'عبد العزيز',         # 'abd' precedes many names
    'في الرحمن', 'الرحمن الرحيم',
]
from collections import defaultdict
raw = defaultdict(int); preceders = defaultdict(set)
for line in corpus:
    toks = line.split()
    for i, w in enumerate(toks):
        raw[w] += 1
        if i > 0: preceders[w].add(toks[i-1])
print(f"{'word':10} {'raw':>4} {'continuation (distinct preceders)':>34}")
for w in sorted(raw, key=lambda x: -raw[x]):
    print(f'{w:10} {raw[w]:4} {len(preceders[w]):>34}')
print()
print('Kneser-Ney backs off on the continuation count, not raw frequency: a word seen')
print('after many different words is the safer guess in an unseen context.')

## 5. Takeaways

Report the normalization script with every WER/CER; prefer subword tokenization for Arabic;
the HMM sums (forward) or maximizes (Viterbi) over alignments; smoothing rewards words that
appear in many contexts.

## Exercise solutions

Exercises 2 and 3 are programmatic and solved here with the functions defined above; Exercises 1, 4, and 5 are design questions, summarized.

**Exercise 2.** Compute the Word Error Rate before and after normalization for the Alef/Ta-marbuta example.

In [ ]:
ref='ذهب الطالب إلى المدرسة'; hyp='ذهب الطالب الى المدرسه'
print('raw  WER: {:.0f}%'.format(100*wer(ref,hyp)))
print('norm WER: {:.0f}%'.format(100*wer(normalize_ar(ref),normalize_ar(hyp))))
print('=> only the scoring convention changed, not the system.')

**Exercise 3.** Tokenize a sentence at word, character, and subword levels and give the counts.

In [ ]:
sent=['وسيكتبونها','للمكتبة']
chars=[c for w in sent for c in w]
bpe=['w','sy','ktbwn','ha','ll','mktba']
for name,t in [('words',sent),('characters',chars),('BPE subwords',bpe)]:
    print(f'{name:14} count={len(t):2}  {t}')
print('=> subwords sit between words (OOV risk) and characters (long sequences).')

**Exercise 1 (design).** The acoustic score is near-tied between عَمّان and عُمان (both written عمان); the language score breaks the tie from context. **Exercise 4 (design).** Rebalance the 8 Gulf hours (upsample or reweight), hold out a speaker-disjoint Gulf set, and report per-dialect WER. **Exercise 5 (design).** وللمكتبة = و + ل + ال + مكتبة; stripping the clitics exposes a shared stem and cuts out-of-vocabulary forms; a toolkit such as CAMeL Tools produces a comparable split (report tool, version, model, scheme).